In [1]:
using SpeedyWeather, CairoMakie, GLMakie

# Define grid and model type
spectral_grid = SpectralGrid()
model = PrimitiveWetModel(spectral_grid)

# Initialise the model and check its ouput
simulation = initialize!(model)
model.output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ humid: specific humidity [kg/kg]
 ├ temp: temperature [degC]
 ├ u: zonal wind [m/s]
 ├ mslp: mean sea-level pressure [hPa]
 └ vor: relative vorticity [s^-1]

In [2]:
# Add radiation and surface flux to the model
add!(model, SpeedyWeather.RadiationOutput()...)
add!(model, SpeedyWeather.SurfaceFluxesOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ slf: Surface latent heat flux (positive up) [W/m^2]
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ shf: Surface sensible heat flux (po

In [3]:
# Check radiation and surface flux types
simulation.diagnostic_variables.physics.sensible_heat_flux
simulation.diagnostic_variables.physics.surface_latent_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 ⋮
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0

In [4]:
# Run simulation
run!(simulation, period=Day(30))

In [5]:
field = simulation.diagnostic_variables.physics.sensible_heat_flux
g  = field.grid
vals = Array(field)   # the 1D field values (length 3168)
heatmap(field)

In [6]:
field

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 46.88108f0
 39.860634f0
 31.37547f0
 22.205515f0
 14.143892f0
  8.885732f0
  6.2358828f0
  4.863484f0
  3.9429836f0
  3.5111647f0
  ⋮
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0

In [7]:
full_field = RingGrids.interpolate(FullGaussianGrid, field)
full_field_matrix = Matrix(full_field)

96×48 Matrix{Float32}:
 46.8811   35.3062     4.86354    2.14995     …   0.0      0.0       0.0  0.0
 45.4185   35.2112     2.09855    9.26381         0.0      0.0       0.0  0.0
 43.9559   35.1163    -0.666434  16.3777          0.0      0.0       0.0  0.0
 42.4933   35.0214    -3.43142   23.4915          0.0      0.0       0.0  0.0
 41.0307   34.9264    -2.82363   24.365           0.0      0.0       0.0  0.0
 39.5071   32.8518     0.313749  25.2386      …   0.0      0.0       0.0  0.0
 37.7393   30.7771     3.45113   26.1121          0.0      0.0       0.0  0.0
 35.9716   28.7024     6.31058   22.6816          0.0      0.0       0.0  0.0
 34.2039   26.6277     7.50244   19.2512          0.0      0.0       0.0  0.0
 32.4361   22.6122     8.69431   15.8208          0.0      0.0       0.0  0.0
  ⋮                                           ⋱            ⋮              
 55.7884  -23.4748     0.0        0.0            29.9356   3.2086    0.0  0.0
 54.9679  -28.0964     0.0        0.0       

In [8]:
#lons, lats = RingGrids.get_londlatds(full_field)

# assume you already have:
# full_field_matrix (matrix) and full_field (interpolated grid object)
sz = size(full_field_matrix)
nlat, nlon = sz

# get lon/lat (whatever shape RingGrids returns)
lons, lats = RingGrids.get_londlatds(full_field)

@show typeof(lons), size(lons), ndims(lons), length(lons)
@show typeof(lats), size(lats), ndims(lats), length(lats)
@show "nlat,nlon" => (nlat,nlon)

lons_grid = reshape(lons, nlat, nlon)
lats_grid = reshape(lats, nlat, nlon)

@show size(lons_grid), size(lats_grid), size(full_field_matrix)


(typeof(lons), size(lons), ndims(lons), length(lons)) = (Vector{Float64}, (4608,), 1, 4608)
(typeof(lats), size(lats), ndims(lats), length(lats)) = (Vector{Float64}, (4608,), 1, 4608)
"nlat,nlon" => (nlat, nlon) = "nlat,nlon" => (96, 48)
(size(lons_grid), size(lats_grid), size(full_field_matrix)) = ((96, 48), (96, 48), (96, 48))


((96, 48), (96, 48), (96, 48))

In [9]:
using CairoMakie, GeoMakie

lons_fixed = map(x -> x > 180 ? x - 360 : x, lons_grid)

fig = Figure(size = (1200, 600))

# GeoAxis: sets up a geographic axis so coastlines and lon/lat align.
ga = GeoAxis(fig[1, 1]; xlabel = "Longitude", ylabel = "Latitude")

# Plot the surface
surface!(ga, lons_fixed, lats_grid, full_field_matrix;
         colormap = :plasma,
         depth_shift = 0.1,  # flat surface 
         transparency = true,
         shading=false)    # allow alpha
         #alpha = 0.1)           

# Draw coastlines on top
lines!(ga, GeoMakie.coastlines(); linewidth = 2, color=:black, depth_shift = 1e-10)

# Add colourbar
Colorbar(fig[1, 2], label = "sensible heat flux", 
         colormap = :plasma, 
         limits = extrema(full_field_matrix))
fig


In [83]:
# Add a heatmap to see global patterns
heatmap(simulation.diagnostic_variables.physics.sensible_heat_flux)
#simulation.prognostic_variables.clock